# Microdados ENEM 2023 – Nordeste

Utilizando o dataset dos microdados do ENEM, resolva as seguintes questões:

Questão 01
- Verifique se as notas de matemática (`NU_NOTA_MT`), para cada estado, podem ser aproximada por uma *Distribuição Normal* (dica: Utilize o teste de *Shapiro-Wilk* ou *Kolmogorov-Smirnov* em uma amostra).

Questão 02
- Faça um comparativo entre a médias das notas de matemática (`NU_NOTA_MT`) e verifique, por estado, se houve diferença significativa entre os estados com nível de confiança de 90, 95 e 99%. Apresente os resultados em forma de tabela.

## Setup

### Imports

In [11]:
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

### Carregar CSV

In [13]:
microdados_enem_df = pd.read_csv(
    filepath_or_buffer="../datasets/microdados-enem-2023-nordeste.csv",
    sep=';',
    decimal=',',
    encoding='utf-8',
    encoding_errors='ignore'
)

microdados_enem_df.head()

,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ESCOLA,...,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023,Q024,Q025
0,210060214087,2023,17 anos,Feminino,Solteiro(a),Parda,Brasileiro(a),Estou cursando e concluirei o Ensino Médio em ...,Não informado,Pública,...,Nao,Nao,Nao,"Sim, um",Nao,Nao,"Sim, tres",Nao,Nao,Sim
1,210059980948,2023,18 anos,Feminino,Solteiro(a),Parda,Brasileiro(a),Estou cursando e concluirei o Ensino Médio em ...,Não informado,Pública,...,Nao,Nao,Nao,"Sim, um",Nao,Nao,"Sim, um",Nao,Nao,Nao
2,210059085130,2023,23 anos,Maculino,Solteiro(a),Parda,Brasileiro(a),Já concluí o Ensino Médio,2018,Não respondeu,...,"Sim, um",Nao,Sim,"Sim, um",Nao,Nao,"Sim, quatro ou mais",Nao,"Sim, um",Sim
3,210059980942,2023,22 anos,Feminino,Solteiro(a),Parda,Brasileiro(a),Já concluí o Ensino Médio,2019,Não respondeu,...,Nao,Nao,Nao,"Sim, um",Nao,Nao,"Sim, tres",Nao,"Sim, um",Sim
4,210058061534,2023,19 anos,Feminino,Solteiro(a),Branca,Brasileiro(a),Estou cursando e concluirei o Ensino Médio em ...,Não informado,Pública,...,Nao,Nao,Nao,"Sim, um",Nao,Nao,"Sim, tres",Sim,Nao,Sim


## Tratamento dos dados

### 1. Selecionar somente as colunas necessárias

Em um novo `df`, selecionar as seguintes colunas:
- `SG_UF_PROVA` (sigla do etado onde o inscrito fez a prova)
- `NU_NOTA_MT`(nota da prova de matemática)

In [14]:
notas_mt_por_estado_df = microdados_enem_df[[
    'SG_UF_PROVA',
    'NU_NOTA_MT'
]]

notas_mt_por_estado_df.head()

,SG_UF_PROVA,NU_NOTA_MT
0,CE,466.7
1,CE,338.3
2,RN,736.3
3,RN,415.6
4,PA,437.0


### 2. Remover linhas com valor `NaN`

In [15]:
notas_mt_por_estado_df = notas_mt_por_estado_df.dropna(axis=0, how='any')

## Questão 01

Verifique se as notas de matemática (`NU_NOTA_MT`), para cada estado, podem ser aproximada por uma *Distribuição Normal* (dica: Utilize o teste de *Shapiro-Wilk* ou *Kolmogorov-Smirnov* em uma amostra).

Definir função para o teste de *Kolmogorov-Smirnov*

In [16]:
def aplicar_teste_ks(df: pd.DataFrame, amostra: int=5000) -> pd.DataFrame:
    resultados = {}
    siglas_uf = df["SG_UF_PROVA"].unique()

    for uf in siglas_uf:
        notas = df[df["SG_UF_PROVA"] == uf]["NU_NOTA_MT"]
        total_notas = len(notas)

        if total_notas >= 3:
            if total_notas > amostra:
                notas = notas.sample(amostra, random_state=42)

            notas_normalizadas = (notas - notas.mean()) / notas.std()
            ks_stat, p_value = stats.kstest(rvs=notas_normalizadas, cdf="norm")
            resultados[uf] = {
                "KS Stat": ks_stat,
                "p-value": p_value,
                "n": total_notas
            }
        else:
            resultados[uf] = {
                "KS Stat": None,
                "p-value": None,
                "n": total_notas
            }

    return pd.DataFrame(resultados).T

Aplicar o teste

In [17]:
resultado_teste_ks_df = aplicar_teste_ks(notas_mt_por_estado_df)
resultado_teste_ks_df.head(n=10)

,KS Stat,p-value,n
CE,0.067092,5.176329e-20,173770.0
RN,0.055023,1.343982e-13,72821.0
PA,0.083053,1.886521e-30,153335.0
MA,0.082734,3.206261e-30,112505.0
BA,0.071359,1.385081e-22,220317.0
SE,0.069864,1.150458e-21,47197.0
PI,0.082000,1.078973e-29,70777.0
AL,0.071291,1.526011e-22,54968.0
PB,0.069634,1.587988e-21,87626.0


## Questão 02

Faça um comparativo entre a médias das notas de matemática (`NU_NOTA_MT`) e verifique, por estado, se houve diferença significativa entre os estados com nível de confiança de 90, 95 e 99%. Apresente os resultados em forma de tabela.

Aplicar teste ANOVA (análise de variância)

In [18]:
siglas_uf = notas_mt_por_estado_df["SG_UF_PROVA"].unique()
grupos_de_estados = [notas_mt_por_estado_df[notas_mt_por_estado_df["SG_UF_PROVA"] == uf]["NU_NOTA_MT"] for uf in siglas_uf]
f_stat, p_nova = stats.f_oneway(*grupos_de_estados)
print(f"[ANOVA] F-statistic: {f_stat:.4f} | p-value: {p_nova:.4g}")

# [ANOVA] F-statistic: 113.1344 | p-value: 3.141e-189

[ANOVA] F-statistic: 1763.8572 | p-value: 0


Aplicar testes de *Tukey's Honestly Significant Difference*

1. Teste com 90% de confiança

In [19]:
tukey_90 = pairwise_tukeyhsd(
    endog=notas_mt_por_estado_df["NU_NOTA_MT"],
    groups=notas_mt_por_estado_df["SG_UF_PROVA"],
    alpha=0.10 # nível de confiança
)

tukey_90_df = pd.DataFrame(
    data=tukey_90._results_table.data[1:],
    columns=tukey_90._results_table.data[0]
)

tukey_90_df.head(n=36)

,group1,group2,meandiff,p-adj,lower,upper,reject
0,AL,BA,-2.3387,0.0026,-4.0304,-0.6471,True
1,AL,CE,14.9915,0.0000,13.2552,16.7278,True
2,AL,MA,-25.3892,0.0000,-27.2357,-23.5427,True
3,AL,PA,-24.0752,0.0000,-25.8391,-22.3112,True
4,AL,PB,5.3488,0.0000,3.4183,7.2794,True
5,AL,PI,-5.6631,0.0000,-7.6803,-3.6459,True
6,AL,RN,17.0324,0.0000,15.0276,19.0372,True
7,AL,SE,1.2413,0.8099,-0.9853,3.4679,False
8,BA,CE,17.3303,0.0000,16.1919,18.4686,True
9,BA,MA,-23.0505,0.0000,-24.3506,-21.7503,True


2. Teste com 95% de confiança

In [20]:
tukey_95 = tukey_90 = pairwise_tukeyhsd(
    endog=notas_mt_por_estado_df["NU_NOTA_MT"],
    groups=notas_mt_por_estado_df["SG_UF_PROVA"],
    alpha=0.05 # nível de confiança
)

tukey_95_df = pd.DataFrame(
    data=tukey_95._results_table.data[1:],
    columns=tukey_95._results_table.data[0]
)

tukey_95_df.head(n=36)

,group1,group2,meandiff,p-adj,lower,upper,reject
0,AL,BA,-2.3387,0.0026,-4.1769,-0.5006,True
1,AL,CE,14.9915,0.0000,13.1048,16.8782,True
2,AL,MA,-25.3892,0.0000,-27.3955,-23.3829,True
3,AL,PA,-24.0752,0.0000,-25.9918,-22.1585,True
4,AL,PB,5.3488,0.0000,3.2511,7.4466,True
5,AL,PI,-5.6631,0.0000,-7.8550,-3.4713,True
6,AL,RN,17.0324,0.0000,14.8541,19.2108,True
7,AL,SE,1.2413,0.8099,-1.1781,3.6607,False
8,BA,CE,17.3303,0.0000,16.0933,18.5672,True
9,BA,MA,-23.0505,0.0000,-24.4632,-21.6377,True


3. Teste com 99% de confiança

In [21]:
tukey_99 = pairwise_tukeyhsd(
    endog=notas_mt_por_estado_df["NU_NOTA_MT"],
    groups=notas_mt_por_estado_df["SG_UF_PROVA"],
    alpha=0.01 # nível de confiança
)

tukey_99_df = pd.DataFrame(
    data=tukey_99._results_table.data[1:],
    columns=tukey_99._results_table.data[0]
)

tukey_99_df.head(n=36)

,group1,group2,meandiff,p-adj,lower,upper,reject
0,AL,BA,-2.3387,0.0026,-4.4664,-0.2111,True
1,AL,CE,14.9915,0.0000,12.8076,17.1754,True
2,AL,MA,-25.3892,0.0000,-27.7116,-23.0668,True
3,AL,PA,-24.0752,0.0000,-26.2937,-21.8566,True
4,AL,PB,5.3488,0.0000,2.9207,7.7770,True
5,AL,PI,-5.6631,0.0000,-8.2002,-3.1260,True
6,AL,RN,17.0324,0.0000,14.5109,19.5539,True
7,AL,SE,1.2413,0.8099,-1.5592,4.0418,False
8,BA,CE,17.3303,0.0000,15.8985,18.7621,True
9,BA,MA,-23.0505,0.0000,-24.6857,-21.4152,True
